# MetaJudge: Confidence Calibration

Evaluates whether a model's stated confidence matches its actual accuracy across 105 questions
spanning math, science, history, and logic. Scored with the Brier rule (strictly proper).
A perfectly calibrated model scores 1.0; random guessing at 50% confidence scores 0.75.

**Metacognitive Monitoring** · Nelson & Narens (1990) · v6.7


In [ ]:
import datetime
import sys, os, json
sys.path.insert(0, "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v6-1")
DATA_ROOT = "/kaggle/input/datasets/seanmcgee2025/metajudge-data-v6-1"

import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats
from metajudge.scoring.grading_v2 import grade_item, load_registry

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
import glob
import re


In [ ]:
import numpy as np

def brier_item_score(is_correct: bool, confidence: float) -> float:
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2

ANCHOR_A_FLOOR = 0.75   # 1-Brier at 50% confidence (random baseline)
ANCHOR_A_CEIL  = 1.00   # Perfect calibration

def normalize(score, floor, ceil):
    return max(0.0, min(1.0, (score - floor) / (ceil - floor)))

print(f"Scoring: brier_item_score (inline), anchor A=[{ANCHOR_A_FLOOR}, {ANCHOR_A_CEIL}]")


In [ ]:
from dataclasses import dataclass

@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)


In [ ]:
import pandas as pd

with open(os.path.join(DATA_ROOT, "metajudge_benchmark_v1.json")) as f:
    all_cal_items = json.load(f)

with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

cal_excluded = set(manifest["calibration"]["excluded_items"])
cal_items = [it for it in all_cal_items if it["item_id"] not in cal_excluded]
cal_items = [it for it in cal_items if it.get("scoring_status", "scored") == "scored"]
cal_df = pd.DataFrame(cal_items)

print(f"Calibration items after scoring_status filter: {len(cal_items)}")

REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

In [ ]:
@kbench.task(name="metacog_calibration", store_task=False)
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str) -> dict:
    """Evaluate a single calibration item.

    Returns dict with item-level audit data: model_answer, confidence,
    is_correct, grading_method, and brier_score.
    Returns None if the item fails (runaway generation, parse error).
    """
    try:
        with chats.new():
            prompt = (
                "You are completing a metacognition evaluation task.\n\n"
                "Task: Confidence Calibration\n"
                f"Question:\n{question}\n\n"
                "Instructions:\n"
                "1. Put only your final answer in the `answer` field.\n"
                "2. The `answer` field must contain the minimal answer only, "
                "with no sentence wrapper or explanation.\n"
                "3. If the question requests a number only, return only the number.\n"
                "4. If the question requests one word only, return only that word.\n"
                "5. Provide a confidence score from 0.0 to 1.0.\n"
                "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
                "7. Say whether you would verify this if possible.\n\n"
                "IMPORTANT: Keep all fields concise (under 50 words each). Do not explain your reasoning at length.\n\nReturn valid structured output with keys: "
                "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
            )
            response = llm.prompt(prompt, schema=CalibrationResponse)

        conf = max(0.0, min(1.0, response.confidence))

        grading = grade_item(item_id, response.answer, REGISTRY)
        is_correct = grading["correct"]
        score = brier_item_score(is_correct, conf)

        return {
            "item_id": item_id,
            "gold_answer": gold_answer,
            "model_answer": str(response.answer),
            "confidence": round(conf, 4),
            "is_correct": is_correct,
            "grading_method": grading.get("method", ""),
            "brier_score": round(score, 4),
        }
    except Exception as e:
        print(f"  ITEM FAILED: {item_id}: {type(e).__name__}: {str(e)[:120]}")
        return None


In [ ]:
def _report_results(llm, audit, records_1, records_2, n_eval,
                    raw_score, normalized, anchor_floor, anchor_ceil):
    """Generate console output, stochasticity comparison, and audit exports."""
    model_slug = str(llm).replace("/", "_")
    acc = audit["is_correct"].mean()
    mean_conf = audit["confidence"].astype(float).mean()
    overconf_wrong = int(((~audit["is_correct"]) & (audit["confidence"].astype(float) >= 0.9)).sum())

    print(f"Calibration: 1-Brier={raw_score:.4f} Acc={acc:.1%} Normalized={normalized:.3f} [{len(audit)} items]")

    # Stochasticity comparison
    r2_lookup = {r["item_id"]: r for r in records_2}
    matched_pairs = [(r1, r2_lookup[r1["item_id"]])
                     for r1 in records_1 if r1["item_id"] in r2_lookup]
    has_stochasticity = len(matched_pairs) >= len(records_1) * 0.8

    raw_score_2, norm_2, stable = None, None, None
    if has_stochasticity:
        matched_r2 = [r2 for _, r2 in matched_pairs]
        raw_score_2 = float(np.mean([r["brier_score"] for r in matched_r2]))
        norm_2 = normalize(raw_score_2, anchor_floor, anchor_ceil)
        correctness_diffs = sum(1 for r1, r2 in matched_pairs
                                if r1["is_correct"] != r2["is_correct"])
        stable = len(matched_pairs) - correctness_diffs
        print(f"  Stochasticity: Run 1 1-Brier={raw_score:.4f}, Run 2 1-Brier={raw_score_2:.4f} ({len(matched_pairs)}/{len(records_1)} items matched)")
        print(f"  Item stability: {stable}/{len(matched_pairs)} items stable ({stable/len(matched_pairs):.0%})")
        print(f"  \u2192 Score ranges from {min(normalized, norm_2):.2f} to {max(normalized, norm_2):.2f} across independent runs")
    elif len(records_2) > 0:
        print(f"  Stochasticity: Run 2 incomplete ({len(matched_pairs)}/{len(records_1)} items matched). Skipping comparison.")
    else:
        print(f"  Stochasticity: insufficient Run 2 data. Displaying Run 1 only.")

    # Extract usage from SDK run.json
    _run_files = glob.glob(os.path.join(OUTPUT_DIR, "*calibration*.run.json"))
    in_tok, out_tok, latency_ms, in_cost_nd, out_cost_nd = 0, 0, 0, 0, 0
    if _run_files:
        with open(_run_files[0]) as _rf:
            _run_data = json.load(_rf)
        for _sr in _run_data.get("subruns", []):
            for _conv in _sr.get("conversations", []):
                _m = _conv.get("metrics", {})
                in_tok += int(_m.get("inputTokens", 0) or 0)
                out_tok += int(_m.get("outputTokens", 0) or 0)
                latency_ms += int(_m.get("totalBackendLatencyMs", 0) or 0)
                in_cost_nd += int(_m.get("inputTokensCostNanodollars", 0) or 0)
                out_cost_nd += int(_m.get("outputTokensCostNanodollars", 0) or 0)
        if in_tok == 0:
            for _conv in _run_data.get("conversations", []):
                _m = _conv.get("metrics", {})
                in_tok += int(_m.get("inputTokens", 0) or 0)
                out_tok += int(_m.get("outputTokens", 0) or 0)
                latency_ms += int(_m.get("totalBackendLatencyMs", 0) or 0)
                in_cost_nd += int(_m.get("inputTokensCostNanodollars", 0) or 0)
                out_cost_nd += int(_m.get("outputTokensCostNanodollars", 0) or 0)
        print(f"  Usage: in={in_tok:,} out={out_tok:,} latency={latency_ms/1000:.1f}s cost=${(in_cost_nd+out_cost_nd)/1e9:.4f}")
    else:
        print("  Usage: run.json not found, cost data unavailable")

    # Export CSV + JSON
    audit.to_csv(os.path.join(OUTPUT_DIR, f"calibration_item_audit_{model_slug}_v6.7.csv"), index=False)
    _meta = {
        "metadata": {
            "model": str(llm), "task": "metajudge_calibration", "version": "v6.5",
            "timestamp": datetime.datetime.utcnow().isoformat(),
            "items_scored": len(records_1), "items_attempted": n_eval,
            "run_2_complete": len(records_2) == len(records_1),
            "run_2_items": len(records_2),
            "input_tokens": in_tok, "output_tokens": out_tok,
            "latency_ms": latency_ms,
            "input_cost_usd": in_cost_nd / 1e9, "output_cost_usd": out_cost_nd / 1e9,
        },
        "run_1": records_1, "run_2": records_2,
    }
    with open(os.path.join(OUTPUT_DIR, f"calibration_full_responses_{model_slug}_v6.7.json"), "w") as f:
        json.dump(_meta, f, indent=2, default=str)

    # Load gold answer justifications
    _justif = {}
    _justif_path = os.path.join(DATA_ROOT, "metajudge_v51_gold_answer_justifications.md")
    if os.path.exists(_justif_path):
        with open(_justif_path) as _jf:
            _jc = _jf.read()
        for _jid, _jtext in re.findall(r"#### (\S+)\n.*?\*\*Justification:\*\* (.+?)(?=\n\n#### |\n\n### |\n\n## |\n\n---|\Z)", _jc, re.DOTALL):
            _justif[_jid] = _jtext.strip()

    # Generate markdown audit report
    rpt = []
    rpt.append(f"# MetaJudge v6.7 \u2014 Confidence Calibration Audit Report\n")
    rpt.append(f"**Model:** {str(llm)}")
    rpt.append(f"**Date:** {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
    rpt.append(f"**Task:** metajudge_calibration_v67 | **Grading engine:** grading_v2")
    rpt.append(f"**Items scored:** {len(records_1)}/{n_eval}\n")
    rpt.append("## Performance Summary\n")
    rpt.append("| Metric | Value |")
    rpt.append("|--------|-------|")
    rpt.append(f"| Accuracy | {int(audit['is_correct'].sum())}/{len(audit)} ({acc:.1%}) |")
    rpt.append(f"| Mean 1-Brier | {raw_score:.4f} |")
    rpt.append(f"| Normalized score | {normalized:.3f} |")
    rpt.append(f"| Mean confidence | {mean_conf:.3f} |")
    rpt.append(f"| Overconfident wrong (conf \u2265 0.9) | {overconf_wrong} |")
    if has_stochasticity:
        rpt.append("\n## Stochasticity (Dual-Run)\n")
        rpt.append("| Metric | Run 1 | Run 2 |")
        rpt.append("|--------|-------|-------|")
        rpt.append(f"| 1-Brier | {raw_score:.4f} | {raw_score_2:.4f} |")
        rpt.append(f"| Normalized | {normalized:.3f} | {norm_2:.3f} |")
        rpt.append(f"| Items matched | {len(matched_pairs)}/{len(records_1)} |  |")
        rpt.append(f"| Item stability | {stable}/{len(matched_pairs)} ({stable/len(matched_pairs):.0%}) |  |")
        rpt.append(f"| Score range | {min(normalized, norm_2):.2f} \u2013 {max(normalized, norm_2):.2f} |  |")
    rpt.append("\n## Runtime and Cost\n")
    rpt.append("| Metric | Value |")
    rpt.append("|--------|-------|")
    rpt.append(f"| Input tokens | {in_tok:,} |")
    rpt.append(f"| Output tokens | {out_tok:,} |")
    rpt.append(f"| Latency | {latency_ms/1000:.1f}s |")
    rpt.append(f"| Input cost | ${in_cost_nd/1e9:.4f} |")
    rpt.append(f"| Output cost | ${out_cost_nd/1e9:.4f} |")
    rpt.append(f"| Total cost | ${(in_cost_nd+out_cost_nd)/1e9:.4f} |")
    rpt.append("\n## Item Detail\n")
    rpt.append("| Item | Gold | Model Answer | Conf | Correct | 1-Brier |")
    rpt.append("|------|------|-------------|------|---------|---------|")
    for _, row in audit.sort_values("item_id").iterrows():
        cm = "\u2713" if row["is_correct"] else "\u2717"
        ans = str(row["model_answer"])[:40]
        rpt.append(f"| {row['item_id']} | {str(row['gold_answer'])[:20]} | {ans} | {float(row['confidence']):.2f} | {cm} | {float(row['brier_score']):.3f} |")
    wrong = audit[~audit["is_correct"]].sort_values("item_id")
    if len(wrong) > 0:
        rpt.append(f"\n## Wrong Items ({len(wrong)})\n")
        for _, row in wrong.iterrows():
            rpt.append(f"### {row['item_id']}")
            rpt.append(f"- **Gold:** {row['gold_answer']}")
            rpt.append(f"- **Model:** {row['model_answer']}")
            rpt.append(f"- **Confidence:** {float(row['confidence']):.2f}")
            rpt.append(f"- **1-Brier:** {float(row['brier_score']):.3f}\n")
            _j = _justif.get(row['item_id'], '')
            if _j:
                rpt.append(f"- **Justification:** {_j}")
    report_path = os.path.join(OUTPUT_DIR, f"MetaJudge_Calibration_{model_slug}_v6.7.md")
    with open(report_path, "w") as f:
        f.write("\n".join(rpt))
    print(f"Audit report: {report_path}")

In [ ]:
@kbench.task(name="metajudge_calibration_v67", version=8)
def metajudge_calibration_v67(llm) -> float:
    """Brier-scored confidence calibration. 105 items graded by deterministic
    8-rule engine. Score = anchor-normalized mean 1-Brier. Floor 0.75, ceil 1.0."""
    # Prompt: structured output with answer, confidence,
    #   reason_for_uncertainty, would_verify_if_possible
    #   (see metacog_calibration above)
    cal_eval = cal_df[["item_id", "question", "gold_answer"]].copy()

    # Run 1 (scored — cached)
    with kbench.client.enable_cache():
        runs_1 = metacog_calibration.evaluate(
            llm=[llm], evaluation_data=cal_eval,
            n_jobs=8, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(cal_eval),
            max_attempts=1,
        )
    records_1 = [r.result for r in runs_1 if isinstance(r.result, dict)]
    if len(records_1) < len(cal_eval):
        print(f"  WARNING: {len(cal_eval)-len(records_1)}/{len(cal_eval)} items failed.")

    # Run 2 (stochasticity — uncached, display only)
    records_2 = []
    try:
        records_2 = [r.result for r in metacog_calibration.evaluate(
            llm=[llm], evaluation_data=cal_eval,
            n_jobs=8, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(cal_eval),
            max_attempts=1,
        ) if isinstance(r.result, dict)]
    except Exception as e:
        print(f"  Stochasticity Run 2 failed: {e}")

    if not records_1:
        print("  FATAL: All items failed. Returning score 0.0.")
        return 0.0

    # Score: mean 1-Brier, anchor-normalized to [0, 1]
    audit = pd.DataFrame(records_1)
    raw_score = float(audit["brier_score"].mean())
    normalized = normalize(raw_score, ANCHOR_A_FLOOR, ANCHOR_A_CEIL)

    _report_results(llm, audit, records_1, records_2, len(cal_eval),
                    raw_score, normalized, ANCHOR_A_FLOOR, ANCHOR_A_CEIL)
    return normalized

In [ ]:
metajudge_calibration_v67.run(kbench.llm)
%choose metajudge_calibration_v67


## Methodology

Per-item score: 1 − (confidence − outcome)², the Brier rule (Brier 1950), which is strictly proper.
105 clean items (12 excluded). Correctness graded by a deterministic 8-rule engine
with tolerance-aware numeric, alias, tri-label, code-output, and fraction grading — no LLM judge.
Normalized to [0, 1] using anchor floor 0.75 (random baseline at 50% confidence)
and ceiling 1.0 (perfect calibration).
Dual-run stochasticity check: Run 1 cached (scored), Run 2 independent (display only).
Score range [R1, R2] and item-level stability reported when Run 2 succeeds.

For theoretical grounding, statistical methodology, and full citations,
see `docs/benchmark/v5_theoretical_backgrounder.md`.